In [ ]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

<bound method NDFrame.head of                                               messages
0    [{'role': 'user', 'content': 'intro'}, {'role'...
1    [{'role': 'user', 'content': 'Who is Varad?'},...
2    [{'role': 'user', 'content': 'Tell me about Va...
3    [{'role': 'user', 'content': 'What is Varad li...
4    [{'role': 'user', 'content': 'What makes Varad...
..                                                 ...
106  [{'role': 'user', 'content': 'Are you consciou...
107  [{'role': 'user', 'content': 'How was this AI ...
108  [{'role': 'user', 'content': 'What is Varad's ...
109  [{'role': 'user', 'content': 'Does Varad know ...
110  [{'role': 'user', 'content': 'What does Varad ...

[111 rows x 1 columns]>


In [ ]:
df['messages'][1][1]['content']

"Varad Bendale. 2nd year Information Technology at K J Somaiya Institute of Technology, Mumbai. 9.57 CGPA — yeah that's not a typo. He's the kind of person who learns Random Forest and immediately goes 'cool, let me rebuild this in C++ from scratch with zero libraries.' That tells you everything. Genuinely passionate about AI and ML, not the surface level stuff — the deep, messy, how-does-this-actually-work kind. Wants to make real impact. Currently making me exist, which is either genius or madness, jury's still out."

In [ ]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())

user_messege = []
asistant_reply = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    asistant_reply.append(df['messages'][i][1]['content'])

question_tokens  = []
target_tokens = []

for i in range(len(df['messages'])) :
    user_encods = enc.encode( "<|user|>"  + user_messege[i] ,  allowed_special= allowed)
    asis_encods = enc.encode( "<|assistant|>" +  asistant_reply[i] + "<|end|>"  , allowed_special= allowed )
    tokens = user_encods + asis_encods
    question = torch.tensor(tokens[:-1])
    question_tokens.append(question)
    target    = torch.tensor(tokens[1:])
    target_tokens.append(target)





In [ ]:
print(question_tokens)

In [ ]:
g = torch.Generator()
g.manual_seed(42)

In [ ]:
max_num = max(set(tokens))
print(max_num)

173781


In [ ]:
vocab_size = len(set(tokens))
print(vocab_size)

107


In [ ]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[tok.item()] for tok in question_tokens[i]])
    t_tensor = torch.tensor([stoi[tok.item()] for tok in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [ ]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [ ]:
block_size = 3
xs = []
ys = []

for i in range(len(q_tokens)):
    context = q_tokens[max(0, i-block_size+1) : i+1]
    context = torch.nn.functional.pad(context, (block_size - len(context), 0))
    xs.append(context)
    ys.append(t_tokens[i])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [ ]:
dim = 100

Encods  = torch.randn((vocab_size, dim), requires_grad=True)
W1 = torch.randn((3*dim, dim), requires_grad=True)
W2 = torch.randn((dim, vocab_size), requires_grad=True)
enc = Encods[xs]
update_enc = enc.view(enc.shape[0] , -1 )
hidden = torch.tanh(update_enc @ W1 )
logits = hidden @ W2

In [ ]:
b_gain = torch.ones((1, 100), requires_grad=True)
b_bias = torch.zeros((1, 100), requires_grad=True)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [ ]:
logits.shape


torch.Size([15555, 2381])

In [ ]:
loss = torch.nn.functional.cross_entropy(logits, ys)
print(loss)

tensor(34.8419, grad_fn=<NllLossBackward0>)


In [ ]:
from google.colab import files
uploaded = files.upload()

checkpoint = torch.load('Amika.pt')
Encods      = checkpoint['Encods']
W1     = checkpoint['W1']
W2     = checkpoint['W2']
b_gain = checkpoint['b_gain']
b_bias = checkpoint['b_bias']
stoi   = checkpoint['stoi']
itos   = checkpoint['itos']

vocab_size = len(stoi)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

Saving Amika.pt to Amika (1).pt


In [ ]:
print(parameters)

In [ ]:
batch_size = 512
prev_loss = 100000
for i in range(1):
    ix = torch.randint(0, xs.shape[0], (batch_size,))
    enc  = Encods[xs[ix]]
    update_enc = enc.view(enc.shape[0], -1)
    flow_layer = update_enc @ W1
    flow_layer = b_gain * (flow_layer - flow_layer.mean(0)) / flow_layer.std(0) + b_bias
    hidden     = torch.tanh(flow_layer)
    logits     = hidden @ W2
    loss       = torch.nn.functional.cross_entropy(logits, ys[ix])

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if prev_loss < loss:
        prev_loss = loss
        break
print(loss)

tensor(5.4646, grad_fn=<NllLossBackward0>)


In [ ]:
import torch.nn as nn

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
positions  = nn.Embedding(block_size, 100)
pos_emb = positions(torch.arange(3).to(device))
x = Encods[xs] + pos_emb

In [57]:
print(x.shape)

torch.Size([15555, 3, 100])


In [ ]:
tokenizer = tiktoken.encoding_for_model('gpt-4o')
size = max(len(tokenizer.encode('<|user|>' + user_messege[i] + '<|assistant|>' + asistant_reply[i] + '<|end|>', allowed_special=allowed)) for i in range(len(df)))
print(size)

298


In [ ]:
mask = torch.tril(torch.ones(size, size)).to(device)
mask = mask / torch.sum(mask,1,keepdim = True)


In [58]:
head_quant = 16
query  = nn.Linear(dim , head_quant , bias = False )
key  = nn.Linear(dim , head_quant , bias = False )
value  = nn.Linear(dim , head_quant , bias = False )
q = query(x)
k = key(x)
v = value(x)

In [61]:
print(v.shape)

torch.Size([15555, 3, 16])
